# run21 — 모델링 설계 상세 진단 (run01b 로지스틱·run08b RandomForest)

H1 Uplift 모델(run01b/run08b)의 "결과"만이 아니라 **어떻게 설계했는지** 과정을 보여주기 위한 노트북.
T-learner 구조상 구독자/비구독자 두 개의 arm 모델이 있는데, 이 노트북은 그중 **구독자(Treatment=1) arm**을
대표로 골라 하이퍼파라미터 튜닝 곡선·계수(중요도)·혼동행렬·ROC 곡선까지 진단한다.

결과 이미지는 `model_charts/` 폴더에 저장됨: `model1_logistic_l1.png`, `model2_randomforest.png`

In [1]:
# -*- coding: utf-8 -*-
"""모델링 설계 상세 진단 차트 생성 (run01b/run08b 구독자 arm 기준)"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.size'] = 12

GREEN = '#2f6b4f'; AMBER = '#a85c1c'; BLUE = '#3a5f8a'; RED = '#b3311f'; GREY = '#8a9488'

NUM_MAP = {
    'log_frequency': '구매빈도(로그)', 'log_monetary': '총구매금액(로그)', 'log_aov': '객단가(로그)',
    'log_recency': '최근성·경과일(로그)', 'regularity_cv': '구매주기 변동성', 'reward_usage_rate': '적립금 사용률',
    '나이': '나이', 'preperiod_제철구매비율': '제철구매비율', 'abnormal_account_flag': '이상계정 플래그',
    'reward_ever_used': '적립금 사용이력', 'preperiod_명절선물세트구매여부': '명절선물세트 구매여부',
}
CAT_MAP = {
    'age_band_h4': '연령대', 'gender_h4': '성별', 'region_tier': '배송권역', '가격구간': '가격구간', '결혼_결측표시': '결혼여부',
}
GENDER_MAP = {'F': '여성', 'M': '남성'}


def clean_name(name):
    if name.startswith('num__'):
        key = name[len('num__'):]
        return NUM_MAP.get(key, key)
    if name.startswith('cat__'):
        rest = name[len('cat__'):]
        for col, label in CAT_MAP.items():
            if rest.startswith(col + '_'):
                val = rest[len(col) + 1:]
                val = GENDER_MAP.get(val, val)
                return f"{label}={val}"
        return rest
    return name

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

BASE = "C:/Users/aidan/OneDrive/바탕 화면/종합실습/data/processed/"
OUT = BASE + "model_charts/"
os.makedirs(OUT, exist_ok=True)

tr = pd.read_csv(BASE + "snapshot_train.csv", encoding="utf-8-sig")
ho = pd.read_csv(BASE + "snapshot_holdout.csv", encoding="utf-8-sig")
for d in (tr, ho):
    d["log_frequency"] = np.log1p(d["frequency"])
    d["log_monetary"] = np.log1p(d["monetary"])
    d["log_aov"] = np.log1p(d["aov"])
    d["log_recency"] = np.log1p(d["recency_days"])
    d["결혼_결측표시"] = d["결혼"].fillna("결측")

NUM_FEATS = ["log_frequency", "log_monetary", "log_aov", "log_recency", "regularity_cv",
             "reward_usage_rate", "나이", "preperiod_제철구매비율", "abnormal_account_flag",
             "reward_ever_used", "preperiod_명절선물세트구매여부"]
CAT_FEATS = ["age_band_h4", "gender_h4", "region_tier", "가격구간", "결혼_결측표시"]
ALL_FEATS = NUM_FEATS + CAT_FEATS

pre = ColumnTransformer([("num", StandardScaler(), NUM_FEATS), ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATS)])
train_known = tr[tr["include_in_uplift_model"] == 1].copy()
X_train_all = pre.fit_transform(train_known[ALL_FEATS])
feat_names = pre.get_feature_names_out()
y_train_all = train_known["y_repurchase_14d"].values
t_train_all = train_known["treatment_h4"].values

ho_known = ho[ho["include_in_uplift_model"] == 1].copy()
X_ho_all = pre.transform(ho_known[ALL_FEATS])
y_ho_all = ho_known["y_repurchase_14d"].values
t_ho_all = ho_known["treatment_h4"].values

# 구독자(Treatment=1) arm만 사용
Xs, ys = X_train_all[t_train_all == 1], y_train_all[t_train_all == 1]
Xs_ho, ys_ho = X_ho_all[t_ho_all == 1], y_ho_all[t_ho_all == 1]
print(f"구독자 train {len(ys)}명(양성률 {ys.mean():.2f}), holdout {len(ys_ho)}명(양성률 {ys_ho.mean():.2f})")

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


def make_panel(fig_path, title, tuning_x, tuning_train, tuning_val, best_idx, best_label,
               coef_names, coef_vals, coef_title, y_true, y_prob, xlabel_tuning, xscale='linear'):
    fig = plt.figure(figsize=(20, 5.6))
    gs = gridspec.GridSpec(1, 4, width_ratios=[1, 1, 1, 1], wspace=0.38)

    # 1. 튜닝 곡선
    ax1 = fig.add_subplot(gs[0])
    ax1.plot(tuning_x, tuning_train, 'o-', color=BLUE, label='학습 폴드')
    ax1.plot(tuning_x, tuning_val, 'o--', color=RED, label='검증 폴드')
    ax1.axvline(tuning_x[best_idx], color=GREY, ls=':', lw=1.5)
    ax1.scatter([tuning_x[best_idx]], [tuning_val[best_idx]], s=140, facecolors='none', edgecolors=AMBER, linewidths=2.2, zorder=5)
    ax1.set_xlabel(xlabel_tuning)
    ax1.set_ylabel('ROC-AUC')
    if xscale == 'log':
        ax1.set_xscale('log')
        tick_vals = [1e-3, 1e-2, 1e-1, 1, 10, 100]
        ax1.set_xticks(tick_vals)
        ax1.set_xticklabels(['0.001', '0.01', '0.1', '1', '10', '100'])
    ax1.set_title(f'하이퍼파라미터 튜닝 (5-fold)\n선택: {best_label}', fontsize=12)
    ax1.legend(fontsize=10, loc='lower right')
    ax1.grid(alpha=0.25)

    # 2. 계수/중요도
    ax2 = fig.add_subplot(gs[1])
    colors = [GREEN if v >= 0 else RED for v in coef_vals]
    y_pos = np.arange(len(coef_names))
    ax2.barh(y_pos, coef_vals, color=colors)
    ax2.set_yticks(y_pos)
    ax2.set_yticklabels(coef_names, fontsize=10)
    ax2.invert_yaxis()
    ax2.axvline(0, color=GREY, lw=1)
    ax2.set_title(coef_title, fontsize=12)
    span = (max(coef_vals) - min(coef_vals)) if len(coef_vals) > 1 else 1
    pad = span * 0.06 + 0.02
    for i, v in enumerate(coef_vals):
        ax2.text(v + (pad if v >= 0 else -pad), i, f'{v:+.2f}', va='center',
                  ha='left' if v >= 0 else 'right', fontsize=9)
    ax2.set_xlim(min(coef_vals) - span * 0.32, max(coef_vals) + span * 0.32)

    # 3. 혼동행렬
    ax3 = fig.add_subplot(gs[2])
    y_pred = (y_prob >= 0.5).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    ax3.imshow(cm, cmap='Blues')
    labels = [['TN', 'FP'], ['FN', 'TP']]
    for i in range(2):
        for j in range(2):
            ax3.text(j, i, f"{cm[i, j]}\n({labels[i][j]})", ha='center', va='center',
                      fontsize=14, fontweight='bold',
                      color='white' if cm[i, j] > cm.max() / 2 else 'black')
    ax3.set_xticks([0, 1]); ax3.set_xticklabels(['예측 비재구매', '예측 재구매'])
    ax3.set_yticks([0, 1]); ax3.set_yticklabels(['실제 비재구매', '실제 재구매'])
    ax3.set_title(f'혼동행렬 (Holdout {len(y_true)}건, 기준 0.5)', fontsize=12)

    # 4. ROC
    ax4 = fig.add_subplot(gs[3])
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    ax4.plot(fpr, tpr, color=RED, lw=2.4)
    ax4.plot([0, 1], [0, 1], color=GREY, ls='--', lw=1)
    ax4.fill_between(fpr, tpr, alpha=0.12, color=RED)
    ax4.set_xlabel('False Positive Rate')
    ax4.set_ylabel('True Positive Rate')
    ax4.set_title(f'ROC 곡선 (Holdout)\nAUC = {auc:.4f}', fontsize=12)
    ax4.grid(alpha=0.25)

    fig.suptitle(title, fontsize=17, fontweight='bold', y=1.04)
    plt.tight_layout()
    plt.savefig(fig_path, dpi=150, facecolor='white', bbox_inches='tight')
    plt.close()
    return auc, cm

구독자 train 1626명(양성률 0.41), holdout 1714명(양성률 0.35)


## MODEL 1 — 로지스틱 회귀 · L1 (run01b, 구독자 모델)

5-fold 교차검증으로 정규화 강도 C를 튜닝하고, 최종 모델의 계수·홀드아웃 성능을 확인한다.

In [2]:
# ============ MODEL 1: 로지스틱 L1 (run01b, 구독자 arm) ============
Cs = np.logspace(-3, 2, 15)
train_scores, val_scores = [], []
for C in Cs:
    clf = LogisticRegression(penalty="l1", solver="liblinear", C=C, max_iter=5000)
    res = cross_validate(clf, Xs, ys, cv=cv5, scoring="roc_auc", return_train_score=True)
    train_scores.append(res["train_score"].mean())
    val_scores.append(res["test_score"].mean())

best_idx = int(np.argmax(val_scores))
best_C = Cs[best_idx]
final_logit = LogisticRegression(penalty="l1", solver="liblinear", C=best_C, max_iter=5000)
final_logit.fit(Xs, ys)
coefs = final_logit.coef_[0]
kept = [(f, c) for f, c in zip(feat_names, coefs) if abs(c) > 1e-6]
kept_sorted = sorted(kept, key=lambda x: -abs(x[1]))[:10]
kept_sorted = sorted(kept_sorted, key=lambda x: x[1])  # 그래프용 오름차순
n_dropped = len(feat_names) - len(kept)

y_prob_logit = final_logit.predict_proba(Xs_ho)[:, 1]
auc1, cm1 = make_panel(
    OUT + "model1_logistic_l1.png",
    "MODEL 1 — 로지스틱 회귀 · L1 (run01b, 구독자 모델)",
    Cs, train_scores, val_scores, best_idx, f"C={best_C:.3f}",
    [clean_name(f) for f, c in kept_sorted], [c for f, c in kept_sorted],
    f"회귀계수 · 0이 아닌 {len(kept)}/{len(feat_names)}개",
    ys_ho, y_prob_logit, "C (정규화 강도의 역수)", xscale='log',
)
print(f"MODEL1 완료: best_C={best_C:.4f}, kept={len(kept)}, dropped={n_dropped}, holdout_auc={auc1:.4f}")
print("cm1:\n", cm1)

C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default va

C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default va

C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default va

C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default va

C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default va

...

C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default va

C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default va

C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default va

C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default va

MODEL1 완료: best_C=0.0268, kept=4, dropped=25, holdout_auc=0.8812
cm1:
 [[882 234]
 [121 477]]


## MODEL 2 — RandomForest · GridSearchCV (run08b, 구독자 모델)

run08b와 동일한 그리드(n_estimators/max_depth/min_samples_leaf)로 재현하고, max_depth 축으로 학습-검증 곡선을 그려 과적합 여부를 확인한다.

In [3]:
# ============ MODEL 2: RandomForest (run08b, 구독자 arm) ============
param_grid = {"n_estimators": [200, 400], "max_depth": [3, 5, 7, None], "min_samples_leaf": [10, 30, 50, 100]}
gs = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=1), param_grid, cv=cv5,
                   scoring="roc_auc", n_jobs=-1, return_train_score=True)
gs.fit(Xs, ys)
best_params = gs.best_params_
print(f"RF best_params: {best_params}, best_cv_auc={gs.best_score_:.4f}")

cvres = pd.DataFrame(gs.cv_results_)
fixed = cvres[(cvres["param_n_estimators"] == best_params["n_estimators"]) &
              (cvres["param_min_samples_leaf"] == best_params["min_samples_leaf"])].copy()
depth_order = [3, 5, 7, None]
fixed["depth_label"] = fixed["param_max_depth"].apply(lambda x: "없음" if x is None else str(x))
fixed = fixed.set_index("param_max_depth").loc[depth_order].reset_index()
depth_x = list(range(len(depth_order)))
rf_train = fixed["mean_train_score"].tolist()
rf_val = fixed["mean_test_score"].tolist()
rf_best_idx = int(np.argmax(rf_val))

final_rf = RandomForestClassifier(random_state=42, n_jobs=1, **best_params)
final_rf.fit(Xs, ys)
imp = pd.Series(final_rf.feature_importances_, index=[clean_name(f) for f in feat_names]).sort_values(ascending=False).head(10)
imp_sorted = imp.sort_values()

y_prob_rf = final_rf.predict_proba(Xs_ho)[:, 1]


def make_panel_rf(fig_path, title, tuning_x, tuning_train, tuning_val, best_idx, best_label, xticklabels,
                   coef_names, coef_vals, coef_title, y_true, y_prob):
    fig = plt.figure(figsize=(20, 5.6))
    gs2 = gridspec.GridSpec(1, 4, width_ratios=[1, 1, 1, 1], wspace=0.38)

    ax1 = fig.add_subplot(gs2[0])
    ax1.plot(tuning_x, tuning_train, 'o-', color=BLUE, label='학습 폴드')
    ax1.plot(tuning_x, tuning_val, 'o--', color=RED, label='검증 폴드')
    ax1.scatter([tuning_x[best_idx]], [tuning_val[best_idx]], s=140, facecolors='none', edgecolors=AMBER, linewidths=2.2, zorder=5)
    ax1.set_xticks(tuning_x); ax1.set_xticklabels(xticklabels)
    ax1.set_xlabel('max_depth (n_estimators·min_samples_leaf 고정)')
    ax1.set_ylabel('ROC-AUC')
    ax1.set_title(f'하이퍼파라미터 튜닝 (5-fold)\n선택: {best_label}', fontsize=12)
    ax1.legend(fontsize=10, loc='lower right')
    ax1.grid(alpha=0.25)

    ax2 = fig.add_subplot(gs2[1])
    y_pos = np.arange(len(coef_names))
    ax2.barh(y_pos, coef_vals, color=GREEN)
    ax2.set_yticks(y_pos); ax2.set_yticklabels(coef_names, fontsize=10)
    ax2.invert_yaxis()
    ax2.set_title(coef_title, fontsize=12)
    for i, v in enumerate(coef_vals):
        ax2.text(v, i, f' {v:.3f}', va='center', fontsize=9)

    ax3 = fig.add_subplot(gs2[2])
    y_pred = (y_prob >= 0.5).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    ax3.imshow(cm, cmap='Blues')
    labels = [['TN', 'FP'], ['FN', 'TP']]
    for i in range(2):
        for j in range(2):
            ax3.text(j, i, f"{cm[i, j]}\n({labels[i][j]})", ha='center', va='center',
                      fontsize=14, fontweight='bold', color='white' if cm[i, j] > cm.max() / 2 else 'black')
    ax3.set_xticks([0, 1]); ax3.set_xticklabels(['예측 비재구매', '예측 재구매'])
    ax3.set_yticks([0, 1]); ax3.set_yticklabels(['실제 비재구매', '실제 재구매'])
    ax3.set_title(f'혼동행렬 (Holdout {len(y_true)}건, 기준 0.5)', fontsize=12)

    ax4 = fig.add_subplot(gs2[3])
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    ax4.plot(fpr, tpr, color=RED, lw=2.4)
    ax4.plot([0, 1], [0, 1], color=GREY, ls='--', lw=1)
    ax4.fill_between(fpr, tpr, alpha=0.12, color=RED)
    ax4.set_xlabel('False Positive Rate'); ax4.set_ylabel('True Positive Rate')
    ax4.set_title(f'ROC 곡선 (Holdout)\nAUC = {auc:.4f}', fontsize=12)
    ax4.grid(alpha=0.25)

    fig.suptitle(title, fontsize=17, fontweight='bold', y=1.04)
    plt.tight_layout()
    plt.savefig(fig_path, dpi=150, facecolor='white', bbox_inches='tight')
    plt.close()
    return auc, cm


best_label_rf = f"n_estimators={best_params['n_estimators']}, max_depth={best_params['max_depth']}, min_samples_leaf={best_params['min_samples_leaf']}"
auc2, cm2 = make_panel_rf(
    OUT + "model2_randomforest.png",
    "MODEL 2 — RandomForest · GridSearchCV (run08b, 구독자 모델)",
    depth_x, rf_train, rf_val, rf_best_idx, best_label_rf, fixed["depth_label"].tolist(),
    list(imp_sorted.index), list(imp_sorted.values), "피처 중요도 상위 10",
    ys_ho, y_prob_rf,
)
print(f"MODEL2 완료: best_params={best_params}, holdout_auc={auc2:.4f}")
print("cm2:\n", cm2)

RF best_params: {'max_depth': None, 'min_samples_leaf': 30, 'n_estimators': 400}, best_cv_auc=0.8826


C:\Users\aidan\AppData\Local\Temp\ipykernel_10608\1591308930.py:77: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


MODEL2 완료: best_params={'max_depth': None, 'min_samples_leaf': 30, 'n_estimators': 400}, holdout_auc=0.8815
cm2:
 [[942 174]
 [144 454]]


## 요약 저장

In [4]:
with open(OUT + "summary.txt", "w", encoding="utf-8") as f:
    f.write(f"MODEL1(로지스틱L1) best_C={best_C:.4f}, kept={len(kept)}/{len(feat_names)}, holdout_auc={auc1:.4f}\n")
    f.write(f"kept coefs top10: {kept_sorted}\n\n")
    f.write(f"MODEL2(RandomForest) best_params={best_params}, holdout_auc={auc2:.4f}\n")
    f.write(f"top10 importance: {list(zip(imp_sorted.index[::-1], imp_sorted.values[::-1]))}\n")

print("전체 완료")


전체 완료
